How Async IO works. 
1. as this is a jupiter notebook, there is alway a threvent loop running. as in alway there will be task runninig in, so we are goin tot yse await over here, so that this is next task that will be picked up. 
2. Async.io tells that to sleep for sometime. 

In [1]:
import asyncio

async def hello():
    await asyncio.sleep(2)
    print("hello")

print("before")

await hello() # Do this in Notebook

print("after")

# asyncio.run(hello()) # Do this in .py file


before
hello
after


## 2. async and await

`async` tells Python that a function can perform asynchronous operations.

`await` tells Python:

"Pause this coroutine until this asynchronous operation is ready, but allow the event loop to do other work."

`await` can only be used inside an `async` function.

In [2]:
import asyncio #This is the import we need to do when we work on real time data. 

async def fetch_data():
    print("Fetching data...")

    await asyncio.sleep(2)

    print("Data received!")
    return "Some data"


async def main():
    data = await fetch_data()
    print(data)

await main()

Fetching data...
Data received!
Some data


In [ ]:
import asyncio
import time

#First method defination
async def task_a():
    print("Task A started")
    await asyncio.sleep(2)
    print("Task A finished")

#Second method defination
async def task_b():
    print("Task B started")
    await asyncio.sleep(7)
    print("Task B finished")

#Single co-routine for calling both the method
async def main():
    await task_a()#we await on the task A
    await task_b()#we await on the task B

#Varible to store the data
start = time.perf_counter()

#Waiting on the main function
await main()

#Printing the value
print(f"Time taken: {time.perf_counter() - start:.2f} seconds")

Task A started
Task A finished
Task B started
Task B finished
Time taken: 9.00 seconds


## 4. Concurrency with asyncio.gather()

So far, we waited for Task A to finish before starting Task B.

But if the tasks are independent, we can run them concurrently.

`asyncio.gather()` schedules multiple coroutines and waits for all of them to complete.

This is particularly useful for I/O-bound operations such as:

- API calls
- Database queries
- LLM requests
- File/network operations

In [7]:
import asyncio
import time


async def task_a():
    print("Task A started")
    await asyncio.sleep(2)
    print("Task A finished")

    return "Result A"


async def task_b():
    print("Task B started")
    await asyncio.sleep(1)
    print("Task B finished")

    return "Result B"

#Over here we are not waiting on any one taks to get completed, we are just calling in both at together.
async def main():
    results = await asyncio.gather(
        task_a(),
        task_b(),
    )

    print(results)


start = time.perf_counter()

await main()

# asyncio.run(main())

print(f"Time taken: {time.perf_counter() - start:.2f} seconds")

Task A started
Task B started
Task B finished
Task A finished
['Result A', 'Result B']
Time taken: 2.00 seconds


## 5. The Event Loop

The event loop is the engine behind asyncio.

It keeps track of asynchronous tasks and resumes them when they are ready to make progress.

When one task is waiting for I/O, the event loop can work on another task. You normally don't need to manually manage the event loop.

For standalone Python programs, `asyncio.run()` is the common entry point. But a Jupyter/VS Code notebook kernel is already running inside its own asyncio event loop (that's how the kernel manages cells under the hood). Notebooks support top-level await for exactly this reason — there's already a loop running, so you hook into it instead of starting a new one.

`asyncio.run()` is for plain .py scripts run from the terminal, where no loop exists yet and something has to create one — that's the if `__name__ == "__main__": asyncio.run(main())` pattern.

## 7. Async Generators and async for

An async generator can produce values asynchronously over time.

We use:

    async def
    yield

to create an async generator.

We consume it using:

    async for

This is particularly useful for streaming responses from LLMs or other APIs.

In [ ]:
import asyncio

async def generate_numbers():

    for number in range(5):
        await asyncio.sleep(5)
        yield number #using this we keep this method running. If i not yeild it, the method will return the output.
        #Generating the value lazyly


async def main():

    async for number in generate_numbers(): #if you want to iterate the async for loop, you need to work with the async
        print(number)


await main()

0
1
2
3
4


Making an API call using the HHTPX asyncronus manner

In [11]:
import asyncio
import httpx


async def fetch_post():

    async with httpx.AsyncClient() as client:

        response = await client.get(
            "https://jsonplaceholder.typicode.com/posts/1"
        )

        return response.json()


async def main():
    post = await fetch_post()
    await asyncio.sleep(10)
    print(post)


await main()

{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


How to use async.gather using to call an api 

In [15]:
import asyncio
import httpx

#This code shows suppose in case you want to call the 
async def fetch_user(client):
    response = await client.get(
        "https://jsonplaceholder.typicode.com/users/1"
    )
    #print(f"Call initated from user method")
    #await asyncio.sleep(5)
    return response.json()


async def fetch_post(client):
    response = await client.get(
        "https://jsonplaceholder.typicode.com/posts/1"
    )
    #print(f"Call initated from post method")
    #await asyncio.sleep(10)
    return response.json()


async def fetch_todo(client):
    response = await client.get(
        "https://jsonplaceholder.typicode.com/todos/1"
    )
    #print(f"Call initated from todos method")
    #await asyncio.sleep(15)
    return response.json()


async def main():

    async with httpx.AsyncClient() as client:

        user, post, todo = await asyncio.gather(
            fetch_user(client),
            fetch_post(client),
            fetch_todo(client),
        )

    print("User:")
    print(user["name"])

    print("\nPost:")
    print(post["title"])

    print("\nTodo:")
    print(todo["title"])


await main()

User:
Leanne Graham

Post:
sunt aut facere repellat provident occaecati excepturi optio reprehenderit

Todo:
delectus aut autem


# 📝 Homework

## Homework 1: Async Functions and `await`

Create an async function called `process_task()` that simulates a task taking 2 seconds to complete.

Inside the function:

1. Print `"Task started"`
2. Wait asynchronously for 2 seconds using `asyncio.sleep()`
3. Print `"Task completed"`
4. Return `"Success"`

Create an `async main()` function that calls `process_task()` and prints the returned result.

### Expected Output

```text
Task started
Task completed
Success

```

### Goal

Understand:

* `async def`
* `await`
* `asyncio.sleep()`
* How an async function is executed using `asyncio.run()`

---

In [19]:
import asyncio 

async def process_task():
    print("Task Started")

    await asyncio.sleep(2)

    print("Task Completed")
    return "Success"


async def main():
    data = await process_task()
    print(data)

await main()

Task Started
Task Completed
Success


## Homework 2: Concurrency with `asyncio.gather()`

You have three independent tasks:

* Downloading a document → 3 seconds
* Generating embeddings → 2 seconds
* Fetching metadata → 1 second

Create three async functions that simulate these operations using `asyncio.sleep()`.

Then use `asyncio.gather()` to execute all three tasks concurrently.

Measure the total execution time using `time.perf_counter()`.

### Requirements

Print the result of each task and the total execution time.

### Expected Observation

If the tasks were executed sequentially, the total time would be approximately:

```text
3 + 2 + 1 = 6 seconds

```

With `asyncio.gather()`, the total time should be approximately:

```text
3 seconds

```

### Goal

Understand how asynchronous concurrency can reduce the total waiting time for independent I/O-bound operations.

---

In [21]:
import asyncio
import time


async def download_document():
    print("Downloading Document Started")
    await asyncio.sleep(3)
    print("Downloading Document finished")

    return "Document Download Success"


async def generating_embadings():
    print("Generating Embeddings started")
    await asyncio.sleep(2)
    print("Generating Enbeddings finished")

    return "Generating Embeddings Success"

async def fetching_metadata():
    print("Fetching Metadata started")
    await asyncio.sleep(1)
    print("Fetching Metadata finished")

    return "Fetching Metadata Success"


async def main():
    results = await asyncio.gather(
        download_document(),
        generating_embadings(),
        fetching_metadata()
    )

    print(results)


start = time.perf_counter()

await main()

# asyncio.run(main())

print(f"Time taken: {time.perf_counter() - start:.2f} seconds")

Generating Embeddings started
Fetching Metadata started
Fetching Metadata finished
Generating Enbeddings finished
['Document Download Success', 'Generating Embeddings Success', 'Fetching Metadata Success']
Time taken: 3.00 seconds


## Homework 3: Concurrent API Requests

Use `httpx.AsyncClient` and the JSONPlaceholder API.

Fetch the following three resources concurrently:

```text
https://jsonplaceholder.typicode.com/users/1
https://jsonplaceholder.typicode.com/posts/1
https://jsonplaceholder.typicode.com/todos/1

```

### Requirements

1. Create an async function called `fetch_data(url)`.
2. Use `httpx.AsyncClient` to make the HTTP request.
3. Return the JSON response.
4. Use `asyncio.gather()` to call all three URLs concurrently.
5. Print the important information from each response.

For example:

```text
User: Leanne Graham
Post ID: 1
Todo: delectus aut autem

```

### Goal

Combine the concepts you have learned:

* `async def`
* `await`
* `httpx.AsyncClient`
* `asyncio.gather()`
* Concurrent I/O

---

In [22]:
import asyncio
import httpx

#This code shows suppose in case you want to call the 
async def fetch_user(client):
    response = await client.get(
        "https://jsonplaceholder.typicode.com/users/1"
    )
    #print(f"Call initated from user method")
    #await asyncio.sleep(5)
    return response.json()


async def fetch_post(client):
    response = await client.get(
        "https://jsonplaceholder.typicode.com/posts/1"
    )
    #print(f"Call initated from post method")
    #await asyncio.sleep(10)
    return response.json()


async def fetch_todo(client):
    response = await client.get(
        "https://jsonplaceholder.typicode.com/todos/1"
    )
    #print(f"Call initated from todos method")
    #await asyncio.sleep(15)
    return response.json()


async def main():

    async with httpx.AsyncClient() as client:

        user, post, todo = await asyncio.gather(
            fetch_user(client),
            fetch_post(client),
            fetch_todo(client),
        )

    print("User:")
    print(user["name"])

    print("\nPost:")
    print(post["title"])

    print("\nTodo:")
    print(todo["title"])


await main()

User:
Leanne Graham

Post:
sunt aut facere repellat provident occaecati excepturi optio reprehenderit

Todo:
delectus aut autem


Imagine that your AI application needs information from three independent sources before calling an LLM:

* User profile
* User's posts
* User's todos

Use the JSONPlaceholder API to simulate these three data sources.

Create three async functions:

```python
async def get_user():
    ...

async def get_posts():
    ...

async def get_todos():
    ...

```

Each function should make an HTTP request using `httpx.AsyncClient`.

Then create an `async main()` function that:

1. Starts all three requests concurrently using `asyncio.gather()`.
2. Waits for all three responses.
3. Combines the results into a single Python dictionary.

The final structure should look similar to:

```python
{
    "user": {...},
    "posts": [...],
    "todos": [...]
}

```

Print the final dictionary.

### Bonus

Measure the total execution time and compare it with a sequential implementation.

### Goal

Think about how async concurrency is used in real AI systems, where an application may need to fetch multiple pieces of information before sending a request to an LLM.

In [ ]:
import asyncio
import time
import httpx


BASE_URL = "https://jsonplaceholder.typicode.com"


async def get_user(client: httpx.AsyncClient, user_id: int = 1):
    response = await client.get(f"{BASE_URL}/users/{user_id}")
    response.raise_for_status()
    return response.json()


async def get_posts(client: httpx.AsyncClient, user_id: int = 1):
    response = await client.get(f"{BASE_URL}/posts?userId={user_id}")
    response.raise_for_status()
    return response.json()


async def get_todos(client: httpx.AsyncClient, user_id: int = 1):
    response = await client.get(f"{BASE_URL}/todos?userId={user_id}")
    response.raise_for_status()
    return response.json()


async def fetch_concurrently(user_id: int = 1):
    async with httpx.AsyncClient() as client:
        user, posts, todos = await asyncio.gather(
            get_user(client, user_id),
            get_posts(client, user_id),
            get_todos(client, user_id),
        )

    return {"user": user, "posts": posts, "todos": todos}


async def fetch_sequentially(user_id: int = 1):
    async with httpx.AsyncClient() as client:
        user = await get_user(client, user_id)
        posts = await get_posts(client, user_id)
        todos = await get_todos(client, user_id)

    return {"user": user, "posts": posts, "todos": todos}


async def main():
    print("=== Concurrent fetch ===")
    start = time.perf_counter()
    result = await fetch_concurrently(user_id=1)
    elapsed = time.perf_counter() - start

    print(f"Total time: {elapsed:.3f}s")
    print(f"User:  {result['user']['name']}")
    print(f"Posts: {len(result['posts'])} posts")
    print(f"Todos: {len(result['todos'])} todos")
    print()

    print("=== Sequential fetch ===")
    start = time.perf_counter()
    result = await fetch_sequentially(user_id=1)
    elapsed = time.perf_counter() - start

    print(f"Total time: {elapsed:.3f}s")
    print(f"User:  {result['user']['name']}")
    print(f"Posts: {len(result['posts'])} posts")
    print(f"Todos: {len(result['todos'])} todos")


await main()

=== Concurrent fetch ===
Total time: 0.543s
User:  Leanne Graham
Posts: 10 posts
Todos: 20 todos

=== Sequential fetch ===
Total time: 0.786s
User:  Leanne Graham
Posts: 10 posts
Todos: 20 todos
